# Notebook 17 — abordagens radicais: transdução + base generativa (base 311)

**Contexto:** data-sculpting esgotado (NB09–NB16, vereditos honestos no NB16:
relabel/quarentena eram canal de vazamento; teto honesto ~0.504–0.509, sub7 publica 0.51).
Duas vias nunca tentadas: **(A)** usar as features do teste (transducao/self-training,
sem labels do teste) e **(B)** uma base *generativa* (erros descorrelacionados das
arvores) para diversidade no stack.
**Protocolo:** mesmas 5 folds seed 42; validacao sempre com rotulos originais.

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
K, TARGET, RANDOM_STATE = 16, 'AGE_RANGE', 42
DATA_DIR = '../../data' if os.path.isdir('../../data') else ('../data' if os.path.isdir('../data') else 'data')
tr = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = tr[TARGET].to_numpy().astype(int)
COLS = [c for c in tr.columns if c not in ('PID', TARGET)]
Xv = tr[COLS].to_numpy(dtype=np.float64)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(Xv, y))
def xgb_recipe(sd=42):
    return XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', random_state=sd,
        n_jobs=-1, eval_metric='mlogloss')
RESULTADOS = []
print('base:', Xv.shape)


## B — Naive Bayes generativo sobre as categóricas cruas + blend com XGB-311

Todas as bases até hoje são árvores discriminativas (erros correlacionados: E8/E12b/NB19b
diluem). NB sobre as ~200 colunas de texto do train_nb01 (routing incluído como
categoria) erra *diferente*. Sozinho deve ser fraco; o teste é o blend com a OOF
XGB-311 (nb11_oof311.npz, F1 0.4782). Duas variantes de prior, sem mais configs.

In [ ]:
B = {}
t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
v01 = pd.read_csv(os.path.join(DATA_DIR, 'test_nb01.csv'), keep_default_na=False, na_values=[''])
pid = tr['PID'].to_numpy()
assert (pd.to_numeric(t01['PID']).to_numpy() == pd.to_numeric(pid)).all(), 'ordem PID difere'
txt_cols = [c for c in t01.select_dtypes(include='str').columns if c != 'PID']
allv = pd.concat([t01[txt_cols], v01[txt_cols]], ignore_index=True).fillna('_FALTA')
Cmat = np.column_stack([pd.factorize(allv[c])[0] for c in txt_cols]).astype(np.int64)
Ctr, Cte = Cmat[:len(t01)], Cmat[len(t01):]
ncats = [int(Cmat[:, j].max()) + 1 for j in range(Cmat.shape[1])]  # espaco global de categorias
print('cats:', Cmat.shape, '| ncols texto:', len(txt_cols))
from sklearn.naive_bayes import CategoricalNB
zx = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))
proba_x, oof_x = zx['proba'], zx['oof'].astype(int)
base311 = f1_score(y, oof_x, average='macro')
NBPROBA = {}
for tag, prior in [('empirico', True), ('uniforme', False)]:
    oof = np.empty(len(y), dtype=int); pro = np.zeros((len(y), K))
    for trn, va in FOLDS:
        m = CategoricalNB(fit_prior=prior, min_categories=ncats)
        m.fit(Ctr[trn], y[trn]); oof[va] = m.predict(Ctr[va]); pro[va] = m.predict_proba(Ctr[va])
    NBPROBA[tag] = pro
    f_nb = f1_score(y, oof, average='macro')
    f_bl = f1_score(y, ((pro + proba_x) / 2).argmax(1), average='macro')
    B[tag] = {'f1_nb': round(float(f_nb), 4), 'f1_blend': round(float(f_bl), 4)}
    RESULTADOS.append({'exp': 'B', 'variante': 'NB-' + tag, 'f1': round(float(f_nb), 4),
                       'nota': 'sozinho'})
    RESULTADOS.append({'exp': 'B', 'variante': 'NB-' + tag + '+XGB311', 'f1': round(float(f_bl), 4),
                       'delta': round(float(f_bl - base311), 4), 'nota': 'blend meio-a-meio'})
    print(tag + ': NB %.4f | blend %.4f (base311 %.4f)' % (f_nb, f_bl, base311))
np.savez(os.path.join(DATA_DIR, 'nb17_nb_oof.npz'), **{'proba_' + t: NBPROBA[t] for t in NBPROBA})
print('OOFs NB guardadas')


## A — simulacao honesta de self-training (transducao)

Por fold externo j: o fold j faz de "teste nao rotulado". Dentro do treino (4 folds),
treina-se em 3, pseudo-rotula-se o top-25% mais confiante do 4.o, re-treina-se em
3+pseudo e avalia-se NO fold externo j (rotulos nunca tocados). Baseline: mesmo
protocolo sem pseudo-labels. Uma so config, sem tuning.

In [ ]:
A = {}
inner_cv = StratifiedKFold(4, shuffle=True, random_state=43)
for j, (a, b) in enumerate(FOLDS):
    # inner split DENTRO do treino externo a: 3 folds treino + 1 fold nao-rotulado.
    # itr/iun sao subconjuntos de a por construcao -> nunca tocam no fold externo b.
    inner = list(inner_cv.split(Xv[a], y[a]))
    iu_local = j % 4
    itr = a[np.concatenate([inner[f][0] for f in range(4) if f != iu_local])]
    iun = a[inner[iu_local][1]]
    assert len(np.intersect1d(itr, b)) == 0 and len(np.intersect1d(iun, b)) == 0
    m0 = xgb_recipe(); m0.fit(Xv[itr], y[itr], sample_weight=compute_sample_weight('balanced', y[itr]))
    pr = m0.predict_proba(Xv[iun]); conf = pr.max(1)
    sel = iun[np.argsort(conf)[-int(len(iun) * 0.25):]]
    Xtr2 = np.vstack([Xv[itr], Xv[sel]]); ytr2 = np.concatenate([y[itr], pr.argmax(1)[np.argsort(conf)[-int(len(iun) * 0.25):]]])
    m1 = xgb_recipe(); m1.fit(Xtr2, ytr2, sample_weight=compute_sample_weight('balanced', ytr2))
    f0 = f1_score(y[b], m0.predict(Xv[b]), average='macro')
    f1 = f1_score(y[b], m1.predict(Xv[b]), average='macro')
    A[j] = {'base3': round(float(f0), 4), 'pseudo': round(float(f1), 4)}
    print(f'fold {j}: base3 {f0:.4f} | +pseudo {f1:.4f}', flush=True)
f0m = np.mean([v['base3'] for v in A.values()]); f1m = np.mean([v['pseudo'] for v in A.values()])
RESULTADOS.append({'exp': 'A', 'variante': 'self-train top25%', 'f1': round(float(f1m), 4),
                   'delta': round(float(f1m - f0m), 4),
                   'nota': f'vs base-3folds {f0m:.4f}, folds ganhos {(sum(1 for v in A.values() if v["pseudo"] > v["base3"] ))}/5'})
print('MEDIA:', round(f0m, 4), '->', round(f1m, 4))
pd.DataFrame(RESULTADOS).to_csv(os.path.join(DATA_DIR, 'nb17_resultados.csv'), index=False)
print('guardado data/nb17_resultados.csv')


In [ ]:
# P1 — confirmacao 5-fold do T10 (pairwise gated, 15 pares, base 311)
# Mesmo protocolo das OOFs (seed 42); referencia: XGB-311 0.4782.
import os as _os
P1P = os.path.join(DATA_DIR, 'nb17_p1.npz')
if _os.path.exists(P1P):
    _z = np.load(P1P); _f1s = list(_z['f1s']); _nov = int(_z['nov'].item())
else:
    from sklearn.utils import compute_sample_weight as _csw
    _mb = xgb_recipe(); _oof_b = np.empty(len(y), dtype=int); _pro_b = np.zeros((len(y), K))
    for _tr, _va in FOLDS:
        _mb.fit(Xv[_tr], y[_tr], sample_weight=_csw('balanced', y[_tr]))
        _oof_b[_va] = _mb.predict(Xv[_va]); _pro_b[_va] = _mb.predict_proba(Xv[_va])
    _top2 = np.sort(np.argsort(_pro_b, axis=1)[:, -2:], axis=1)
    _pred = _oof_b.copy(); _nov = 0
    for a_, b_ in [(k, k + 1) for k in range(15)]:
        for _tr, _va in FOLDS:
            mtr = np.isin(y[_tr], [a_, b_]); yb = (y[_tr][mtr] == b_).astype(int)
            m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                              subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                              random_state=42, n_jobs=-1, eval_metric='logloss')
            m.fit(Xv[_tr][mtr], yb, sample_weight=_csw('balanced', yb))
            sel = (_top2[_va, 0] == a_) & (_top2[_va, 1] == b_)
            sv = _va[sel]
            if len(sv):
                _pred[sv] = np.where(m.predict_proba(Xv[sv])[:, 1] >= 0.5, b_, a_)
                _nov += int(len(sv))
    _f1s = [float(f1_score(y[_va], _pred[_va], average='macro')) for _, _va in FOLDS]
    np.savez(P1P, f1s=np.array(_f1s), oof=_pred, nov=np.array([_nov]))
    print('P1 calculado e guardado', flush=True)
import pandas as _pd
print('P1 pairwise 15 pares 5-fold: %.4f +/- %.4f override %d' % (np.mean(_f1s), np.std(_f1s), _nov), flush=True)
print('folds:', np.round(_f1s, 4).tolist(), flush=True)
_r = _pd.DataFrame([{'exp': 'P1', 'variante': 'pairwise-15 5-fold', 'f1': round(float(np.mean(_f1s)), 4),
                     'std': round(float(np.std(_f1s)), 4), 'nota': 'override %d; base xgb311 0.4782' % _nov}])
_p = os.path.join(DATA_DIR, 'nb17_resultados.csv')
_prev = _pd.read_csv(_p)
_prev = _prev[~_prev['variante'].isin(_r['variante'])]
_pd.concat([_prev, _r], ignore_index=True).to_csv(_p, index=False)


In [ ]:
# P2 — confirmacao 5-fold da C3 (target-mean suavizado + pairwise-15, base 311)
import os as _os
P2P = os.path.join(DATA_DIR, 'nb17_p2.npz')
if _os.path.exists(P2P):
    _z = np.load(P2P); _f1s = list(_z['f1s']); _f0 = list(_z['f0']); _nov = int(_z['nov'].item())
else:
    from sklearn.utils import compute_sample_weight as _csw
    t01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), keep_default_na=False, na_values=[''])
    assert (pd.to_numeric(t01['PID']).to_numpy() == pd.to_numeric(tr['PID'].to_numpy())).all()
    _drop = [c for c in COLS if any(c.startswith(p + '_') for p in ['MARSTALEGAL', 'MAINSTAT'])]
    _keep = [c for c in COLS if c not in _drop]
    Xb = tr[_keep].to_numpy(dtype=np.float64)
    _pred = np.empty(len(y), dtype=int); _base = np.empty(len(y), dtype=int); _nov = 0
    for _tr, _va in FOLDS:
        _glob = y[_tr].mean()
        _tm_tr = np.zeros((len(_tr), 3)); _tm_va = np.zeros((len(_va), 3))
        for j_, c_ in enumerate(['MARSTALEGAL', 'MAINSTAT', 'HATLEVEL']):
            _g = pd.Series(y[_tr]).groupby(t01[c_].iloc[_tr].values)
            _sm = ((_g.sum() + 100.0 * _glob) / (_g.size() + 100.0))
            _tm_tr[:, j_] = t01[c_].iloc[_tr].map(_sm).fillna(_glob).to_numpy()
            _tm_va[:, j_] = t01[c_].iloc[_va].map(_sm).fillna(_glob).to_numpy()
        Xtr = np.hstack([Xb[_tr], _tm_tr]); Xva = np.hstack([Xb[_va], _tm_va])
        _mb = xgb_recipe(); _mb.fit(Xtr, y[_tr], sample_weight=_csw('balanced', y[_tr]))
        _pb = _mb.predict_proba(Xva); _pa = _mb.predict(Xva)
        _base[_va] = _pa
        _top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
        pv = _pa.copy()
        for a_, b_ in [(k, k + 1) for k in range(15)]:
            mtr = np.isin(y[_tr], [a_, b_]); yb = (y[_tr][mtr] == b_).astype(int)
            m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                              subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                              random_state=42, n_jobs=-1, eval_metric='logloss')
            m.fit(Xtr[mtr], yb, sample_weight=_csw('balanced', yb))
            sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
            sv = np.where(sel)[0]
            if len(sv):
                pv[sv] = np.where(m.predict_proba(Xva[sv])[:, 1] >= 0.5, b_, a_)
                _nov += int(len(sv))
        _pred[_va] = pv
    _f1s = [float(f1_score(y[_va], _pred[_va], average='macro')) for _, _va in FOLDS]
    _f0 = [float(f1_score(y[_va], _base[_va], average='macro')) for _, _va in FOLDS]
    np.savez(P2P, f1s=np.array(_f1s), f0=np.array(_f0), oof=_pred, nov=np.array([_nov]))
    print('P2 calculado e guardado', flush=True)
import pandas as _pd
print('P2 C3 5-fold done', _nov, flush=True)
_r = _pd.DataFrame([{'exp': 'P2', 'variante': 'C3-targetmean+pairwise 5-fold', 'f1': round(float(np.mean(_f1s)), 4),
                     'std': round(float(np.std(_f1s)), 4), 'nota': 'override %d; base-TM %.4f' % (_nov, float(np.mean(_f0)))}])
_p = os.path.join(DATA_DIR, 'nb17_resultados.csv')
_prev = _pd.read_csv(_p)
_prev = _prev[~_prev['variante'].isin(_r['variante'])]
_pd.concat([_prev, _r], ignore_index=True).to_csv(_p, index=False)
print('folds:', np.round(_f1s, 4).tolist(), flush=True)


In [ ]:
# P3 — confirmacao 5-fold da C4 (base monotonica + pairwise-15, base 311)
import os as _os
P3P = os.path.join(DATA_DIR, 'nb17_p3.npz')
if _os.path.exists(P3P):
    _z = np.load(P3P); _f1s = list(_z['f1s']); _f0 = list(_z['f0']); _nov = int(_z['nov'].item())
else:
    from sklearn.utils import compute_sample_weight as _csw
    _mc = {c: 0 for c in COLS}
    for c in ['N_DOENCAS', 'N_DIFICULDADES', 'IDADE_MINIMA', 'FAIXA_MINIMA']:
        if c in _mc: _mc[c] = 1
    if 'HATLEVEL' in _mc: _mc['HATLEVEL'] = -1
    _mono = tuple(_mc[c] for c in COLS)
    _pred = np.empty(len(y), dtype=int); _base = np.empty(len(y), dtype=int); _nov = 0
    for _tr, _va in FOLDS:
        _mb = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                            colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                            monotone_constraints=_mono, random_state=42,
                            n_jobs=-1, eval_metric='mlogloss')
        _mb.fit(Xv[_tr], y[_tr], sample_weight=_csw('balanced', y[_tr]))
        _pb = _mb.predict_proba(Xv[_va]); _pa = _mb.predict(Xv[_va])
        _base[_va] = _pa
        _top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
        pv = _pa.copy()
        for a_, b_ in [(k, k + 1) for k in range(15)]:
            mtr = np.isin(y[_tr], [a_, b_]); yb = (y[_tr][mtr] == b_).astype(int)
            m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                              subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                              random_state=42, n_jobs=-1, eval_metric='logloss')
            m.fit(Xv[_tr][mtr], yb, sample_weight=_csw('balanced', yb))
            sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_)
            sv = np.where(sel)[0]
            if len(sv):
                pv[sv] = np.where(m.predict_proba(Xv[_va][sv])[:, 1] >= 0.5, b_, a_)
                _nov += int(len(sv))
        _pred[_va] = pv
    _f1s = [float(f1_score(y[_va], _pred[_va], average='macro')) for _, _va in FOLDS]
    _f0 = [float(f1_score(y[_va], _base[_va], average='macro')) for _, _va in FOLDS]
    np.savez(P3P, f1s=np.array(_f1s), f0=np.array(_f0), oof=_pred, nov=np.array([_nov]))
    print('P3 calculado e guardado', flush=True)
import pandas as _pd
print('P3 C4 5-fold: %.4f +/- %.4f | base-mono %.4f | override %d' % (np.mean(_f1s), np.std(_f1s), float(np.mean(_f0)), _nov), flush=True)
print('folds:', np.round(_f1s, 4).tolist(), flush=True)
_r = _pd.DataFrame([{'exp': 'P3', 'variante': 'C4-mono+pairwise 5-fold', 'f1': round(float(np.mean(_f1s)), 4),
                     'std': round(float(np.std(_f1s)), 4), 'nota': 'override %d; base-mono %.4f' % (_nov, float(np.mean(_f0)))}])
_p = os.path.join(DATA_DIR, 'nb17_resultados.csv')
_prev = _pd.read_csv(_p)
_prev = _prev[~_prev['variante'].isin(_r['variante'])]
_pd.concat([_prev, _r], ignore_index=True).to_csv(_p, index=False)


In [ ]:
# P4 — confirmacao 5-fold da T14 (pairwise-15 com gate pmax<0.6, base 311)
import os as _os
P4P = os.path.join(DATA_DIR, 'nb17_p4.npz')
if _os.path.exists(P4P):
    _z = np.load(P4P); _f1s = list(_z['f1s']); _f0 = list(_z['f0']); _nov = int(_z['nov'].item())
else:
    from sklearn.utils import compute_sample_weight as _csw
    _pred = np.empty(len(y), dtype=int); _base = np.empty(len(y), dtype=int); _nov = 0
    for _tr, _va in FOLDS:
        _mb = xgb_recipe(); _mb.fit(Xv[_tr], y[_tr], sample_weight=_csw('balanced', y[_tr]))
        _pb = _mb.predict_proba(Xv[_va]); _pa = _mb.predict(Xv[_va])
        _base[_va] = _pa
        _top2 = np.sort(np.argsort(_pb, axis=1)[:, -2:], axis=1)
        _maxp = _pb.max(1)
        pv = _pa.copy()
        for a_, b_ in [(k, k + 1) for k in range(15)]:
            mtr = np.isin(y[_tr], [a_, b_]); yb = (y[_tr][mtr] == b_).astype(int)
            m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, min_child_weight=3,
                              subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                              random_state=42, n_jobs=-1, eval_metric='logloss')
            m.fit(Xv[_tr][mtr], yb, sample_weight=_csw('balanced', yb))
            sel = (_top2[:, 0] == a_) & (_top2[:, 1] == b_) & (_maxp < 0.6)
            sv = np.where(sel)[0]
            if len(sv):
                pv[sv] = np.where(m.predict_proba(Xv[_va][sv])[:, 1] >= 0.5, b_, a_)
                _nov += int(len(sv))
        _pred[_va] = pv
    _f1s = [float(f1_score(y[_va], _pred[_va], average='macro')) for _, _va in FOLDS]
    _f0 = [float(f1_score(y[_va], _base[_va], average='macro')) for _, _va in FOLDS]
    np.savez(P4P, f1s=np.array(_f1s), f0=np.array(_f0), oof=_pred, nov=np.array([_nov]))
    print('P4 calculado e guardado', flush=True)
import pandas as _pd
print('P4 T14 5-fold: %.4f +/- %.4f | base %.4f | override %d' % (np.mean(_f1s), np.std(_f1s), float(np.mean(_f0)), _nov), flush=True)
print('folds:', np.round(_f1s, 4).tolist(), flush=True)
_r = _pd.DataFrame([{'exp': 'P4', 'variante': 'T14-gated 5-fold', 'f1': round(float(np.mean(_f1s)), 4),
                     'std': round(float(np.std(_f1s)), 4), 'nota': 'override %d; base %.4f' % (_nov, float(np.mean(_f0)))}])
_p = os.path.join(DATA_DIR, 'nb17_resultados.csv')
_prev = _pd.read_csv(_p)
_prev = _prev[~_prev['variante'].isin(_r['variante'])]
_pd.concat([_prev, _r], ignore_index=True).to_csv(_p, index=False)
